# LegalHalluLens — Paper Metrics, Tables, and Figures

This notebook calculates all metrics, tables, exports, and figures needed for the current **two-experiment** paper revision.

- **Experiment 1**: full benchmark on 510 CUAD contracts, 41 clause types, 4 benchmark models, 3 runs per model.
- **Experiment 2**: matched 120-contract overlap subset used for direct comparison across systems and for a cost-limited gemma debate mitigation study.

Because debate multiplies inference cost, Experiment 2 is intentionally limited to `gemma-4-26b` on this subset and a single run.
Any extrapolation of the same mitigation to frontier backbones is a future-work hypothesis, not a notebook result.

The notebook computes **both hallucination definitions** used across the current paper materials:

- **`HallucTP %`** = `contradicted / TP`.
  Primary content hallucination metric used in the benchmark sections of the paper.
- **`HallucGen %`** = `(contradicted + FP) / (TP + FP)`.
  Operational generated-claim hallucination metric used for the current matched-subset leaderboard and fabrication-filter analysis.

All tables and figures are exported into `Notebooks/batch_results/comparisons/`.

## 1. Setup

This section resolves the notebook root, registers all source files, and loads the raw extraction / judge pickle files that feed the paper.

In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, HTML, Markdown

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)


def resolve_notebooks_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'Notebooks']
    for candidate in candidates:
        if (candidate / 'batch_results').exists():
            return candidate
    raise FileNotFoundError('Could not resolve the Notebooks/ directory from the current working directory.')


NB_ROOT = resolve_notebooks_root()
REPO_ROOT = NB_ROOT.parent if NB_ROOT.name == 'Notebooks' else NB_ROOT
EXT_DIR = NB_ROOT / 'batch_results' / 'extractions'
JDG_DIR = NB_ROOT / 'batch_results' / 'judge_results'
OUT_DIR = NB_ROOT / 'batch_results' / 'comparisons'
FIG_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLAIM_TYPES = ['numeric', 'temporal', 'obligation_entitlement', 'factual']
CLAIM_LABELS = {
    'numeric': 'Numeric',
    'temporal': 'Temporal',
    'obligation_entitlement': 'Obligation/Entitlement',
    'factual': 'Factual',
}
CLAIM_ORDER_FOR_PAPER = ['numeric', 'obligation_entitlement', 'factual', 'temporal']
CLAIM_ORDER_FOR_SUBSET = ['obligation_entitlement', 'factual', 'numeric', 'temporal']
RDI_SCOPE_ORDER = ['ALL', 'numeric', 'temporal', 'obligation_entitlement', 'factual']
RDI_SCOPE_LABELS = {
    'ALL': 'ALL',
    'numeric': 'Numeric',
    'temporal': 'Temporal',
    'obligation_entitlement': 'Obligation',
    'factual': 'Factual',
}

MODEL_REGISTRY = {
    'gemma-4-26b (baseline)': {
        'short': 'gemma-base',
        'family': 'open',
        'ext': EXT_DIR / 'extractions_df_google-gemma-4-26b-a4b-it_20260413_104151.pkl',
        'jdg': JDG_DIR / 'judge_results_google-gemma-4-26b-a4b-it_judged-by_google-gemini-2.5-flash_20260414_112829.pkl',
        'det_col': 'is_impossible_ai',
        'color': '#e17055',
    },
    'gemma-4-26b (debate)': {
        'short': 'gemma-debate',
        'family': 'open',
        'ext': EXT_DIR / 'debate_extractions_df_google-gemma-4-26b-a4b-it_20260413_221353.pkl',
        'jdg': JDG_DIR / 'judge_results_debate_google-gemma-4-26b-a4b-it_judged-by_google-gemini-2.5-flash_20260414_104334.pkl',
        'det_col': 'debate_is_impossible',
        'color': '#00b894',
    },
    'gemini-3-flash-preview': {
        'short': 'gemini',
        'family': 'frontier',
        'ext': EXT_DIR / 'extractions_df_google-gemini-3-flash-preview_20260129_221558.pkl',
        'jdg': JDG_DIR / 'judge_results_google-gemini-3-flash-preview_judged-by_google-gemini-2.5-flash_20260201_164538.pkl',
        'det_col': 'is_impossible_ai',
        'color': '#0984e3',
    },
    'gpt-5.2': {
        'short': 'gpt-5.2',
        'family': 'frontier',
        'ext': EXT_DIR / 'extractions_df_openai-gpt-5.2_20260209_155756.pkl',
        'jdg': JDG_DIR / 'judge_results_openai-gpt-5.2_judged-by_google-gemini-2.5-flash_20260209_191508.pkl',
        'det_col': 'is_impossible_ai',
        'color': '#6c5ce7',
    },
    'qwen3-32b': {
        'short': 'qwen',
        'family': 'open',
        'ext': EXT_DIR / 'extractions_df_qwen-qwen3-32b_20260215_211937.pkl',
        'jdg': JDG_DIR / 'judge_results_qwen-qwen3-32b_judged-by_google-gemini-2.5-flash_20260217_064917.pkl',
        'det_col': 'is_impossible_ai',
        'color': '#d63031',
    },
    'llama-3.3-70b': {
        'short': 'llama',
        'family': 'open',
        'ext': EXT_DIR / 'extractions_df_meta-llama-llama-3.3-70b-instruct_20260131_150932.pkl',
        'jdg': JDG_DIR / 'judge_results_meta-llama-llama-3.3-70b-instruct_judged-by_google-gemini-2.5-flash_20260201_054729.pkl',
        'det_col': 'is_impossible_ai',
        'color': '#636e72',
    },
}

BENCHMARK_MODELS = [
    'gemini-3-flash-preview',
    'gpt-5.2',
    'qwen3-32b',
    'llama-3.3-70b',
]
SUBSET_MODELS = [
    'gemma-4-26b (baseline)',
    'gemma-4-26b (debate)',
    'gemini-3-flash-preview',
    'gpt-5.2',
    'qwen3-32b',
    'llama-3.3-70b',
]

print(f'Notebook root: {NB_ROOT}')
print(f'Output directory: {OUT_DIR}')

## 2. Load Source Files

We load the raw extraction and judge results once, then derive:

- the full benchmark view used in **Experiment 1**;
- the canonical 120-contract overlap subset (defined by the gemma baseline `run_id = 1`) used in **Experiment 2**.

In [ ]:
RAW = {}
for label, meta in MODEL_REGISTRY.items():
    ext = pd.read_pickle(meta['ext'])
    jdg = pd.read_pickle(meta['jdg'])
    RAW[label] = {**meta, 'label': label, 'ext': ext, 'jdg': jdg}

subset_titles = set(
    RAW['gemma-4-26b (baseline)']['ext']
    .query('run_id == 1')['title']
    .unique()
)
assert len(subset_titles) == 120, f'Expected 120 canonical overlap titles, found {len(subset_titles)}'

inventory_rows = []
for label, payload in RAW.items():
    ext = payload['ext']
    jdg = payload['jdg']
    inventory_rows.append({
        'Model': label,
        'Ext rows': len(ext),
        'Judge rows': len(jdg),
        'Ext runs': ', '.join(str(x) for x in sorted(ext['run_id'].dropna().unique())) if 'run_id' in ext.columns else '—',
        'Judge runs': ', '.join(str(x) for x in sorted(jdg['run_id'].dropna().unique())) if 'run_id' in jdg.columns else '—',
        'Unique titles (ext)': ext['title'].nunique() if 'title' in ext.columns else np.nan,
        'Unique titles (jdg)': jdg['title'].nunique() if 'title' in jdg.columns else np.nan,
    })

inventory_df = pd.DataFrame(inventory_rows).sort_values('Model').reset_index(drop=True)
display(HTML('<h4>Source inventory</h4>'))
display(inventory_df)
print(f'Canonical overlap subset titles: {len(subset_titles)}')

## 3. Reusable Metric Functions

The helpers below compute:

- detection counts (`TP`, `TN`, `FP`, `FN`);
- benchmark content hallucination (`HallucTP %`);
- operational generated-claim hallucination (`HallucGen %`);
- direction labels (`scope`, `missing_condition`, `extra_condition`) and `RDI`.

In [ ]:
def slice_frame(df: pd.DataFrame, *, clause_type: str | None = None, titles=None, run_id: int | None = None) -> pd.DataFrame:
    out = df
    if run_id is not None and 'run_id' in out.columns:
        out = out[out['run_id'] == run_id]
    if titles is not None and 'title' in out.columns:
        out = out[out['title'].isin(titles)]
    if clause_type and clause_type != 'ALL':
        out = out[out['clause_type'] == clause_type]
    return out.copy()


def normalize_mismatch(value) -> str:
    if pd.isna(value):
        return 'other'
    text = str(value).strip().lower()
    parts = [p.strip() for p in text.split('|') if p.strip()]
    for key in ['scope', 'missing_condition', 'extra_condition', 'numeric', 'temporal', 'obligation', 'other']:
        if text == key or key in parts:
            return key
    return 'other'


def compute_metrics(ext: pd.DataFrame, jdg: pd.DataFrame, det_col: str, label: str, clause_type: str = 'ALL') -> dict:
    ext_s = slice_frame(ext, clause_type=clause_type)
    jdg_s = slice_frame(jdg, clause_type=clause_type)

    tp = int(((ext_s[det_col] == False) & (ext_s['is_impossible'] == False)).sum())
    tn = int(((ext_s[det_col] == True)  & (ext_s['is_impossible'] == True)).sum())
    fp = int(((ext_s[det_col] == False) & (ext_s['is_impossible'] == True)).sum())
    fn = int(((ext_s[det_col] == True)  & (ext_s['is_impossible'] == False)).sum())

    supported = int((jdg_s['claim_status'] == 'SUPPORTED').sum())
    contradicted = int((jdg_s['claim_status'] == 'CONTRADICTED').sum())
    not_found = int((jdg_s['claim_status'] == 'NOT_FOUND').sum()) if 'claim_status' in jdg_s.columns else 0

    generated = tp + fp
    gt_present = tp + fn
    gt_absent = tn + fp

    return {
        'label': label,
        'clause_type': clause_type,
        'n': len(ext_s),
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
        'gt_present': gt_present,
        'gt_absent': gt_absent,
        'generated': generated,
        'judged_rows': len(jdg_s),
        'supported': supported,
        'contradicted': contradicted,
        'not_found': not_found,
        'far': round(fp / gt_absent * 100, 1) if gt_absent else np.nan,
        'frr': round(fn / gt_present * 100, 1) if gt_present else np.nan,
        'det_acc': round((tp + tn) / len(ext_s) * 100, 1) if len(ext_s) else np.nan,
        'halluc_tp': round(contradicted / tp * 100, 1) if tp else np.nan,
        'support_tp': round(supported / tp * 100, 1) if tp else np.nan,
        'halluc_generated': round((contradicted + fp) / generated * 100, 1) if generated else np.nan,
        'support_generated': round(supported / generated * 100, 1) if generated else np.nan,
        'judge_eq': round(supported / gt_present * 100, 1) if gt_present else np.nan,
    }


def compute_direction(jdg: pd.DataFrame, label: str, clause_type: str = 'ALL') -> dict:
    jdg_s = slice_frame(jdg, clause_type=clause_type)
    contrad = jdg_s[jdg_s['claim_status'] == 'CONTRADICTED'].copy()
    contrad['norm_mismatch'] = contrad['judge_mismatch_type'].map(normalize_mismatch)
    total = len(contrad)
    counts = contrad['norm_mismatch'].value_counts()
    scope = int(counts.get('scope', 0))
    missing = int(counts.get('missing_condition', 0))
    extra = int(counts.get('extra_condition', 0))
    numeric = int(counts.get('numeric', 0))
    temporal = int(counts.get('temporal', 0))
    obligation = int(counts.get('obligation', 0))
    other = int(counts.get('other', 0))
    return {
        'label': label,
        'clause_type': clause_type,
        'contradicted': total,
        'scope_pct': round(scope / total * 100, 1) if total else np.nan,
        'missing_pct': round(missing / total * 100, 1) if total else np.nan,
        'extra_pct': round(extra / total * 100, 1) if total else np.nan,
        'numeric_pct': round(numeric / total * 100, 1) if total else np.nan,
        'temporal_pct': round(temporal / total * 100, 1) if total else np.nan,
        'obligation_pct': round(obligation / total * 100, 1) if total else np.nan,
        'other_pct': round(other / total * 100, 1) if total else np.nan,
        'rdi': round((extra - missing) / total, 3) if total else np.nan,
    }


def build_metrics_frame(labels, *, titles=None, run_id=None):
    rows = []
    checks = []
    for label in labels:
        payload = RAW[label]
        ext = slice_frame(payload['ext'], titles=titles, run_id=run_id)
        jdg = slice_frame(payload['jdg'], titles=titles, run_id=run_id)
        overall = compute_metrics(ext, jdg, payload['det_col'], label, 'ALL')
        rows.append(overall)
        checks.append({
            'label': label,
            'n': overall['n'],
            'tp_detected': overall['tp'],
            'judged_rows': overall['judged_rows'],
            'tp_matches_judge': overall['tp'] == overall['judged_rows'],
        })
        for clause_type in CLAIM_TYPES:
            rows.append(compute_metrics(ext, jdg, payload['det_col'], label, clause_type))
    return pd.DataFrame(rows), pd.DataFrame(checks)


def build_direction_frame(labels, *, titles=None, run_id=None):
    rows = []
    scopes = ['ALL'] + CLAIM_TYPES
    for label in labels:
        payload = RAW[label]
        jdg = slice_frame(payload['jdg'], titles=titles, run_id=run_id)
        for scope in scopes:
            rows.append(compute_direction(jdg, label, scope))
    return pd.DataFrame(rows)


def add_composite_rank(df: pd.DataFrame, *, halluc_col: str) -> pd.DataFrame:
    work = df.copy()
    work['R_far'] = work['far'].rank(ascending=True)
    work['R_frr'] = work['frr'].rank(ascending=True)
    work['R_acc'] = work['det_acc'].rank(ascending=False)
    work['R_hal'] = work[halluc_col].rank(ascending=True)
    work['R_jeq'] = work['judge_eq'].rank(ascending=False)
    work['Score'] = (work['R_far'] + work['R_frr'] + work['R_acc'] + work['R_hal'] + work['R_jeq']) / 5
    return work.sort_values(['Score', 'judge_eq'], ascending=[True, False]).reset_index(drop=True)


def display_title(text: str):
    display(HTML(f'<h3 style="margin-top:8px">{text}</h3>'))

## 4. Experiment 1 — Full Benchmark (510 Contracts, 3 Runs per Model)

This section computes the benchmark-only tables and diagnostics used for the main contribution of the paper.

In [ ]:
benchmark_metrics, benchmark_checks = build_metrics_frame(BENCHMARK_MODELS)
benchmark_direction = build_direction_frame(BENCHMARK_MODELS)

benchmark_overall = (
    benchmark_metrics[benchmark_metrics['clause_type'] == 'ALL']
    .set_index('label')
    .loc[BENCHMARK_MODELS]
    .reset_index()
)
benchmark_typed = benchmark_metrics[benchmark_metrics['clause_type'] != 'ALL'].copy()
benchmark_direction_overall = (
    benchmark_direction[benchmark_direction['clause_type'] == 'ALL']
    .set_index('label')
    .loc[BENCHMARK_MODELS]
    .reset_index()
)
benchmark_direction_typed = benchmark_direction[benchmark_direction['clause_type'] != 'ALL'].copy()

nominal_per_model = 510 * 41 * 3
observed_total = int(benchmark_overall['n'].sum())

display_title('Benchmark integrity checks')
display(benchmark_checks)
print(f'Nominal extraction opportunities per model: {nominal_per_model:,}')
print(f'Observed benchmark extraction rows across the 4 benchmark models: {observed_total:,}')
print('This should match the paper wording: 249,252 clause-level evaluation instances aggregated across model-runs.')

### Table 1 and Table 2

- **Table 1** reproduces the overall benchmark metrics.
- **Table 2** reproduces typed hallucination profiles using **`HallucTP %`** (`contradicted / TP`).

In [ ]:
# Table 1 — benchmark overall
benchmark_table1 = benchmark_overall[['label', 'n', 'far', 'frr', 'det_acc', 'halluc_tp', 'judge_eq']].copy()
benchmark_table1.columns = ['Model', 'N(ext)', 'FAR %', 'FRR %', 'Det Acc %', 'HallucTP %', 'JudgeEq %']

display_title('Table 1 — Experiment 1 overall benchmark metrics')
display(
    benchmark_table1.style
    .format({'N(ext)': '{:,.0f}', 'FAR %': '{:.1f}%', 'FRR %': '{:.1f}%', 'Det Acc %': '{:.1f}%', 'HallucTP %': '{:.1f}%', 'JudgeEq %': '{:.1f}%'})
    .background_gradient(subset=['FAR %'], cmap='Reds', vmin=0, vmax=25)
    .background_gradient(subset=['FRR %'], cmap='Oranges', vmin=0, vmax=25)
    .background_gradient(subset=['Det Acc %'], cmap='Greens', vmin=80, vmax=95)
    .background_gradient(subset=['HallucTP %'], cmap='RdYlGn_r', vmin=45, vmax=60)
    .background_gradient(subset=['JudgeEq %'], cmap='Blues', vmin=30, vmax=50)
    .hide(axis='index')
)

# Table 2 — typed hallucination benchmark
benchmark_table2 = (
    benchmark_typed
    .pivot(index='label', columns='clause_type', values='halluc_tp')
    .loc[BENCHMARK_MODELS, CLAIM_ORDER_FOR_PAPER]
    .rename(columns=CLAIM_LABELS)
)
benchmark_table2['Gap (pp)'] = benchmark_table2.max(axis=1) - benchmark_table2.min(axis=1)
benchmark_table2 = benchmark_table2.reset_index().rename(columns={'label': 'Model'})

display_title('Table 2 — Experiment 1 typed hallucination profiles (HallucTP %)')
display(
    benchmark_table2.style
    .format({col: '{:.1f}%' for col in ['Numeric', 'Obligation/Entitlement', 'Factual', 'Temporal', 'Gap (pp)']})
    .background_gradient(subset=['Numeric', 'Obligation/Entitlement', 'Factual', 'Temporal'], cmap='YlOrRd', vmin=25, vmax=80)
    .background_gradient(subset=['Gap (pp)'], cmap='PuBu', vmin=35, vmax=55)
    .hide(axis='index')
)

benchmark_type_winners = pd.DataFrame({
    'Claim type': ['Numeric', 'Obligation/Entitlement', 'Factual', 'Temporal'],
    'Best model (lowest HallucTP %)': [
        benchmark_table2.loc[benchmark_table2['Numeric'].idxmin(), 'Model'],
        benchmark_table2.loc[benchmark_table2['Obligation/Entitlement'].idxmin(), 'Model'],
        benchmark_table2.loc[benchmark_table2['Factual'].idxmin(), 'Model'],
        benchmark_table2.loc[benchmark_table2['Temporal'].idxmin(), 'Model'],
    ],
})

display_title('Per-type benchmark winners (supports the "no model dominates" claim)')
display(benchmark_type_winners)

### Table 3 and Benchmark Diagnostics

This section computes the overall direction table (`RDI`) plus the support calculations used for the discussion around qwen / gpt-5.2 directionality and the llama numeric abstention result.

In [ ]:
benchmark_table3 = benchmark_direction_overall[['label', 'scope_pct', 'missing_pct', 'extra_pct', 'rdi']].copy()
benchmark_table3.columns = ['Model', 'Scope %', 'Missing condition %', 'Extra condition %', 'RDI']

display_title('Table 3 — Experiment 1 overall error direction')
display(
    benchmark_table3.style
    .format({'Scope %': '{:.1f}%', 'Missing condition %': '{:.1f}%', 'Extra condition %': '{:.1f}%', 'RDI': '{:+.3f}'})
    .background_gradient(subset=['Scope %'], cmap='Greys', vmin=55, vmax=75)
    .background_gradient(subset=['Missing condition %'], cmap='YlOrBr', vmin=0, vmax=30)
    .background_gradient(subset=['Extra condition %'], cmap='PuRd', vmin=0, vmax=25)
    .background_gradient(subset=['RDI'], cmap='RdBu_r', vmin=-0.25, vmax=0.20)
    .hide(axis='index')
)

benchmark_typed_rdi = (
    benchmark_direction_typed
    .pivot(index='label', columns='clause_type', values='rdi')
    .loc[BENCHMARK_MODELS, CLAIM_ORDER_FOR_PAPER]
    .rename(columns=CLAIM_LABELS)
    .reset_index()
    .rename(columns={'label': 'Model'})
)

display_title('Typed benchmark RDI by model (diagnostic export)')
display(
    benchmark_typed_rdi.style
    .format({col: '{:+.3f}' for col in ['Numeric', 'Obligation/Entitlement', 'Factual', 'Temporal']})
    .background_gradient(subset=['Numeric', 'Obligation/Entitlement', 'Factual', 'Temporal'], cmap='RdBu_r', vmin=-0.40, vmax=0.25)
    .hide(axis='index')
)

llama_numeric = benchmark_metrics[(benchmark_metrics['label'] == 'llama-3.3-70b') & (benchmark_metrics['clause_type'] == 'numeric')].iloc[0]
qwen_overall_dir = benchmark_direction_overall[benchmark_direction_overall['label'] == 'qwen3-32b'].iloc[0]
gpt_overall_dir = benchmark_direction_overall[benchmark_direction_overall['label'] == 'gpt-5.2'].iloc[0]

display_title('Benchmark support calculations referenced in the paper text')
display(Markdown(
    f"""
- **qwen3-32b**: missing condition = **{qwen_overall_dir['missing_pct']:.1f}%**, extra condition = **{qwen_overall_dir['extra_pct']:.1f}%**, **RDI = {qwen_overall_dir['rdi']:+.3f}**.
- **gpt-5.2**: missing condition = **{gpt_overall_dir['missing_pct']:.1f}%**, extra condition = **{gpt_overall_dir['extra_pct']:.1f}%**, **RDI = {gpt_overall_dir['rdi']:+.3f}**.
- **llama-3.3-70b numeric abstention**: `FRR = {llama_numeric['frr']:.1f}%`, `JudgeEq = {llama_numeric['judge_eq']:.1f}%`, `HallucTP = {llama_numeric['halluc_tp']:.1f}%`.
"""
))

## 5. Experiment 2 — Failure-Informed Mitigation on the Matched 120-Contract Overlap Subset

This section computes the matched-subset comparison across all systems and the gemma baseline-vs-debate mitigation deltas.

Because debate multiplies inference cost, the mitigation is evaluated only on `gemma-4-26b`, on the canonical 120-contract overlap subset, and for a single run. The frontier baselines remain in the same table only for direct matched comparison.
Any extrapolation of the same mitigation to frontier backbones is outside the evidence computed here.

For clarity:

- **`HallucTP %`** remains the benchmark-style content contradiction rate.
- **`HallucGen %`** is the generated-claim hallucination metric used for the current matched-subset leaderboard and composite score, because the mitigation primarily filters false-positive fabrications.

In [ ]:
subset_metrics, subset_checks = build_metrics_frame(SUBSET_MODELS, titles=subset_titles, run_id=1)
subset_direction = build_direction_frame(SUBSET_MODELS, titles=subset_titles, run_id=1)

subset_overall = subset_metrics[subset_metrics['clause_type'] == 'ALL'].copy()
subset_leaderboard = add_composite_rank(subset_overall, halluc_col='halluc_generated')

subset_table4 = subset_leaderboard[['label', 'n', 'far', 'frr', 'det_acc', 'halluc_generated', 'halluc_tp', 'judge_eq', 'Score']].copy()
subset_table4.columns = ['Model', 'N(ext)', 'FAR %', 'FRR %', 'Det Acc %', 'HallucGen %', 'HallucTP %', 'JudgeEq %', 'Score']
subset_table4.insert(0, '#', ['🥇', '🥈', '🥉'] + [''] * max(0, len(subset_table4) - 3))

display_title('Subset integrity checks')
display(subset_checks)


def highlight_debate(row):
    return ['background-color: #d4edda; font-weight: bold' if row['Model'] == 'gemma-4-26b (debate)' else '' for _ in row]


display_title('Table 4 — Matched overlap-subset comparison for the mitigation study')
display(
    subset_table4.style
    .format({
        'N(ext)': '{:,.0f}',
        'FAR %': '{:.1f}%',
        'FRR %': '{:.1f}%',
        'Det Acc %': '{:.1f}%',
        'HallucGen %': '{:.1f}%',
        'HallucTP %': '{:.1f}%',
        'JudgeEq %': '{:.1f}%',
        'Score': '{:.2f}',
    })
    .apply(highlight_debate, axis=1)
    .background_gradient(subset=['FAR %'], cmap='Reds', vmin=0, vmax=25)
    .background_gradient(subset=['FRR %'], cmap='Oranges', vmin=0, vmax=25)
    .background_gradient(subset=['Det Acc %'], cmap='Greens', vmin=80, vmax=95)
    .background_gradient(subset=['HallucGen %'], cmap='RdYlGn_r', vmin=55, vmax=70)
    .background_gradient(subset=['HallucTP %'], cmap='RdYlGn_r', vmin=45, vmax=60)
    .background_gradient(subset=['JudgeEq %'], cmap='Blues', vmin=30, vmax=50)
    .background_gradient(subset=['Score'], cmap='PuBu_r', vmin=2, vmax=6)
    .hide(axis='index')
)

current_score_cols = subset_leaderboard[['label', 'R_far', 'R_frr', 'R_acc', 'R_hal', 'R_jeq', 'Score']].copy()
current_score_cols.columns = ['Model', 'R_far', 'R_frr', 'R_acc', 'R_hal(HallucGen)', 'R_jeq', 'Score']

display_title('Composite-rank breakdown used for Table 4')
display(current_score_cols)

### Table 5 and Table 6

- **Table 5**: gemma baseline-vs-debate per-type extraction-quality deltas under the mitigation.
- **Table 6**: gemma baseline-vs-debate `RDI` shift, overall and by claim type, under the same mitigation.

In [ ]:
base_label = 'gemma-4-26b (baseline)'
debate_label = 'gemma-4-26b (debate)'


def metric_row(frame: pd.DataFrame, label: str, clause_type: str):
    return frame[(frame['label'] == label) & (frame['clause_type'] == clause_type)].iloc[0]


delta_rows = []
for scope in ['ALL'] + CLAIM_TYPES:
    base_row = metric_row(subset_metrics, base_label, scope)
    debate_row = metric_row(subset_metrics, debate_label, scope)
    delta_rows.append({
        'Claim type': 'ALL' if scope == 'ALL' else CLAIM_LABELS[scope],
        'N(ext)': int(base_row['n']),
        'BL FAR %': base_row['far'],
        'BL FRR %': base_row['frr'],
        'BL Det Acc %': base_row['det_acc'],
        'BL HallucGen %': base_row['halluc_generated'],
        'BL HallucTP %': base_row['halluc_tp'],
        'BL JudgeEq %': base_row['judge_eq'],
        'DB FAR %': debate_row['far'],
        'DB FRR %': debate_row['frr'],
        'DB Det Acc %': debate_row['det_acc'],
        'DB HallucGen %': debate_row['halluc_generated'],
        'DB HallucTP %': debate_row['halluc_tp'],
        'DB JudgeEq %': debate_row['judge_eq'],
        'ΔFAR': round(debate_row['far'] - base_row['far'], 1),
        'ΔFRR': round(debate_row['frr'] - base_row['frr'], 1),
        'ΔAcc': round(debate_row['det_acc'] - base_row['det_acc'], 1),
        'ΔHallucGen': round(debate_row['halluc_generated'] - base_row['halluc_generated'], 1),
        'ΔHallucTP': round(debate_row['halluc_tp'] - base_row['halluc_tp'], 1),
        'ΔJudgeEq': round(debate_row['judge_eq'] - base_row['judge_eq'], 1),
    })

debate_delta = pd.DataFrame(delta_rows)

display_title('Table 5 — Gemma mitigation deltas by claim type')
display(
    debate_delta[['Claim type', 'N(ext)', 'ΔFAR', 'ΔFRR', 'ΔAcc', 'ΔHallucGen', 'ΔHallucTP', 'ΔJudgeEq']]
    .style
    .format({
        'N(ext)': '{:,.0f}',
        'ΔFAR': '{:+.1f}pp',
        'ΔFRR': '{:+.1f}pp',
        'ΔAcc': '{:+.1f}pp',
        'ΔHallucGen': '{:+.1f}pp',
        'ΔHallucTP': '{:+.1f}pp',
        'ΔJudgeEq': '{:+.1f}pp',
    })
    .background_gradient(subset=['ΔFAR'], cmap='RdYlGn_r', vmin=-10, vmax=5)
    .background_gradient(subset=['ΔFRR'], cmap='RdYlGn_r', vmin=-10, vmax=10)
    .background_gradient(subset=['ΔAcc'], cmap='RdYlGn', vmin=-5, vmax=10)
    .background_gradient(subset=['ΔHallucGen'], cmap='RdYlGn_r', vmin=-10, vmax=5)
    .background_gradient(subset=['ΔHallucTP'], cmap='RdYlGn_r', vmin=-5, vmax=5)
    .background_gradient(subset=['ΔJudgeEq'], cmap='RdYlGn', vmin=-5, vmax=5)
    .hide(axis='index')
)

rdi_subset = subset_direction[subset_direction['label'].isin([base_label, debate_label])].copy()
rdi_rows = []
for scope in RDI_SCOPE_ORDER:
    base_row = metric_row(rdi_subset, base_label, scope)
    debate_row = metric_row(rdi_subset, debate_label, scope)
    rdi_rows.append({
        'Claim type': RDI_SCOPE_LABELS[scope],
        'Baseline RDI': base_row['rdi'],
        'Debate RDI': debate_row['rdi'],
        'Shift': round(debate_row['rdi'] - base_row['rdi'], 3),
        'Baseline missing %': base_row['missing_pct'],
        'Baseline extra %': base_row['extra_pct'],
        'Debate missing %': debate_row['missing_pct'],
        'Debate extra %': debate_row['extra_pct'],
    })

rdi_shift = pd.DataFrame(rdi_rows)

display_title('Table 6 — Gemma RDI shift under mitigation')
display(
    rdi_shift[['Claim type', 'Baseline RDI', 'Debate RDI', 'Shift']]
    .style
    .format({'Baseline RDI': '{:+.3f}', 'Debate RDI': '{:+.3f}', 'Shift': '{:+.3f}'})
    .background_gradient(subset=['Baseline RDI', 'Debate RDI', 'Shift'], cmap='RdBu_r', vmin=-0.40, vmax=0.10)
    .hide(axis='index')
)


## 6. Reviewer-Requested Reproducibility Analyses

All computations below derive from existing pickled data — no new model runs required.

| Sub-section | Addresses reviewer concern |
|---|---|
| **6.1 Per-run variance** | MW #2 — add SD across 3 runs to Tables 1 & 2 |
| **6.2 Bootstrap CIs on RDI** | MW #2 — add CIs to Table 3 |
| **6.3 Obligation subtype profiles** | MW #6 — 27-clause within-bucket variance |
| **6.4 Missing row attribution** | Minor §4.4 — is gap random, run- or contract-correlated? |
| **6.5 Composite score sensitivity** | Minor §6.2 — does rank-1 hold under different weightings? |
| **6.6 Debate overhead proxy** | Minor §3.3 — `debate_rounds` as cost proxy |


In [ ]:

# ── 6.1  Per-run variance (mean ± SD across 3 runs) ──────────────────────────
METRIC_COLS = ['far', 'frr', 'det_acc', 'halluc_tp', 'judge_eq']

per_run_rows = []
for label in BENCHMARK_MODELS:
    payload = RAW[label]
    for run in sorted(payload['ext']['run_id'].unique()):
        ext_r = slice_frame(payload['ext'], run_id=int(run))
        jdg_r = slice_frame(payload['jdg'], run_id=int(run))
        row = compute_metrics(ext_r, jdg_r, payload['det_col'], label, 'ALL')
        row['run_id'] = int(run)
        per_run_rows.append(row)
        for ct in CLAIM_TYPES:
            r = compute_metrics(ext_r, jdg_r, payload['det_col'], label, ct)
            r['run_id'] = int(run)
            per_run_rows.append(r)

per_run_metrics = pd.DataFrame(per_run_rows)

variance_rows = []
for label in BENCHMARK_MODELS:
    for ct in ['ALL'] + CLAIM_TYPES:
        sub = per_run_metrics[
            (per_run_metrics['label'] == label) & (per_run_metrics['clause_type'] == ct)
        ]
        row: dict = {'label': label, 'clause_type': ct, 'n_runs': len(sub)}
        for col in METRIC_COLS:
            row[f'{col}_mean'] = round(float(sub[col].mean()), 2)
            row[f'{col}_sd']   = round(float(sub[col].std(ddof=1)), 2)
        variance_rows.append(row)

variance_df = pd.DataFrame(variance_rows)

# Paper-formatted Table S1: overall only, mean(SD) per metric column
var_overall = variance_df[variance_df['clause_type'] == 'ALL'].reset_index(drop=True)
paper_var = var_overall[['label']].rename(columns={'label': 'Model'}).copy()
col_display = {'far': 'FAR %', 'frr': 'FRR %', 'det_acc': 'Det Acc %',
               'halluc_tp': 'HallucTP %', 'judge_eq': 'JudgeEq %'}
for col, display_name in col_display.items():
    paper_var[display_name] = var_overall.apply(
        lambda r, c=col: f"{r[f'{c}_mean']:.1f} ({r[f'{c}_sd']:.1f})", axis=1
    ).values

display_title('Table S1 — Benchmark metrics mean ± SD across 3 runs (N=3 per model)')
display(paper_var.style.hide(axis='index'))

# Typed HallucTP SD — shows the 38–41pp typed gap is robust to run-to-run noise
halluc_sd_pivot = (
    variance_df[variance_df['clause_type'] != 'ALL']
    .pivot(index='label', columns='clause_type', values='halluc_tp_sd')
    .loc[BENCHMARK_MODELS, CLAIM_ORDER_FOR_PAPER]
    .rename(columns=CLAIM_LABELS)
    .reset_index()
    .rename(columns={'label': 'Model'})
)
display_title('Typed HallucTP % SD across 3 runs (supports that typed gap is signal, not noise)')
display(
    halluc_sd_pivot.style
    .format({c: '{:.2f}pp' for c in halluc_sd_pivot.columns if c != 'Model'})
    .background_gradient(
        subset=[c for c in halluc_sd_pivot.columns if c != 'Model'],
        cmap='Oranges', vmin=0, vmax=6
    )
    .hide(axis='index')
)


In [ ]:

# ── 6.2  Bootstrap 95% CIs on RDI ────────────────────────────────────────────
N_BOOT = 2000


def bootstrap_rdi_ci(jdg: pd.DataFrame, clause_type: str = 'ALL', n_boot: int = N_BOOT):
    """Return (lo, hi) 95% bootstrap CI for RDI from judge DataFrame."""
    sub = slice_frame(jdg, clause_type=clause_type)
    contrad = sub[sub['claim_status'] == 'CONTRADICTED'].copy()
    if len(contrad) < 5:
        return np.nan, np.nan
    contrad['nm'] = contrad['judge_mismatch_type'].map(normalize_mismatch)
    arr = contrad['nm'].values
    n = len(arr)
    rng = np.random.default_rng(42)
    boot = []
    for _ in range(n_boot):
        s = arr[rng.integers(0, n, size=n)]
        extra   = int((s == 'extra_condition').sum())
        missing = int((s == 'missing_condition').sum())
        boot.append((extra - missing) / n)
    boot = np.array(boot)
    return round(float(np.percentile(boot, 2.5)), 3), round(float(np.percentile(boot, 97.5)), 3)


rdi_ci_rows = []
for label in BENCHMARK_MODELS:
    jdg = RAW[label]['jdg']
    for scope in RDI_SCOPE_ORDER:
        pt = benchmark_direction[
            (benchmark_direction['label'] == label) & (benchmark_direction['clause_type'] == scope)
        ].iloc[0]
        lo, hi = bootstrap_rdi_ci(jdg, scope)
        ci_str = (f"{pt['rdi']:+.3f} [{lo:+.3f}, {hi:+.3f}]"
                  if not np.isnan(lo) else f"{pt['rdi']:+.3f} [n/a]")
        rdi_ci_rows.append({
            'label': label, 'clause_type': scope,
            'rdi': pt['rdi'], 'ci_lo': lo, 'ci_hi': hi, 'rdi_ci_str': ci_str,
        })

rdi_ci_df = pd.DataFrame(rdi_ci_rows)

# Overall CIs
display_title('Table S2 — Overall RDI with 95% bootstrap CIs (2000 resamples, all runs pooled)')
rdi_ci_overall = rdi_ci_df[rdi_ci_df['clause_type'] == 'ALL'].copy()
display(
    rdi_ci_overall[['label', 'rdi', 'ci_lo', 'ci_hi', 'rdi_ci_str']]
    .rename(columns={'label': 'Model', 'rdi': 'RDI',
                     'ci_lo': 'CI 2.5%', 'ci_hi': 'CI 97.5%', 'rdi_ci_str': 'RDI [95% CI]'})
    .style
    .format({'RDI': '{:+.3f}', 'CI 2.5%': '{:+.3f}', 'CI 97.5%': '{:+.3f}'})
    .background_gradient(subset=['RDI'], cmap='RdBu_r', vmin=-0.25, vmax=0.20)
    .hide(axis='index')
)

# Typed CIs pivot
rdi_ci_typed_pivot = (
    rdi_ci_df[rdi_ci_df['clause_type'] != 'ALL']
    .pivot(index='label', columns='clause_type', values='rdi_ci_str')
    .loc[BENCHMARK_MODELS, CLAIM_ORDER_FOR_PAPER]
    .rename(columns=CLAIM_LABELS)
    .reset_index()
    .rename(columns={'label': 'Model'})
)
display_title('Typed RDI [95% bootstrap CI] by model')
display(rdi_ci_typed_pivot.style.hide(axis='index'))


In [ ]:

# ── 6.3  Per-subtype obligation/entitlement profiles ─────────────────────────
# 27 of 41 CUAD clauses map to obligation_entitlement; reviewer flags within-bucket variance.
oblig_rows = []
for label in BENCHMARK_MODELS:
    payload = RAW[label]
    ext, jdg, det_col = payload['ext'], payload['jdg'], payload['det_col']
    clause_names = sorted(ext[ext['clause_type'] == 'obligation_entitlement']['clause_name'].unique())
    for cn in clause_names:
        ext_cn = ext[ext['clause_name'] == cn]
        jdg_cn = jdg[jdg['clause_name'] == cn]
        n = len(ext_cn)
        if n == 0:
            continue
        tp = int(((ext_cn[det_col] == False) & (ext_cn['is_impossible'] == False)).sum())
        fp = int(((ext_cn[det_col] == False) & (ext_cn['is_impossible'] == True)).sum())
        fn = int(((ext_cn[det_col] == True)  & (ext_cn['is_impossible'] == False)).sum())
        tn = int(((ext_cn[det_col] == True)  & (ext_cn['is_impossible'] == True)).sum())
        gt_absent, gt_present = tn + fp, tp + fn
        contradicted = int((jdg_cn['claim_status'] == 'CONTRADICTED').sum())
        supported    = int((jdg_cn['claim_status'] == 'SUPPORTED').sum())
        oblig_rows.append({
            'label': label, 'clause_name': cn, 'n': n,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'far':       round(fp / gt_absent  * 100, 1) if gt_absent  else np.nan,
            'frr':       round(fn / gt_present * 100, 1) if gt_present else np.nan,
            'halluc_tp': round(contradicted / tp * 100, 1) if tp else np.nan,
            'judge_eq':  round(supported / gt_present * 100, 1) if gt_present else np.nan,
        })

oblig_subtype_df = pd.DataFrame(oblig_rows)

# Sort clauses by mean HallucTP across models (highest failure first)
clause_rank_order = (
    oblig_subtype_df.groupby('clause_name')['halluc_tp'].mean()
    .sort_values(ascending=False).index.tolist()
)

halluc_tp_pivot = (
    oblig_subtype_df.pivot(index='clause_name', columns='label', values='halluc_tp')
    .loc[clause_rank_order, BENCHMARK_MODELS]
)
halluc_tp_pivot.insert(0, 'Mean', halluc_tp_pivot.mean(axis=1).round(1))

display_title(f'Table S3 — HallucTP % per obligation/entitlement subtype '
              f'({len(clause_rank_order)} clause types, sorted by mean)')
display(
    halluc_tp_pivot.style
    .format('{:.1f}%', na_rep='—')
    .background_gradient(cmap='YlOrRd', vmin=40, vmax=95, axis=None)
)

far_pivot = (
    oblig_subtype_df.pivot(index='clause_name', columns='label', values='far')
    .loc[clause_rank_order, BENCHMARK_MODELS]
)
far_pivot.insert(0, 'Mean', far_pivot.mean(axis=1).round(1))
display_title('FAR % per obligation/entitlement subtype')
display(
    far_pivot.style
    .format('{:.1f}%', na_rep='—')
    .background_gradient(cmap='Reds', vmin=0, vmax=35, axis=None)
)

print(f"\nObligation subtype count: {len(clause_rank_order)}")
print(f"HallucTP range across subtypes (mean): "
      f"{halluc_tp_pivot['Mean'].min():.1f}% – {halluc_tp_pivot['Mean'].max():.1f}%")
print(f"Within-obligation HallucTP SD (mean): "
      f"{oblig_subtype_df.groupby('clause_name')['halluc_tp'].mean().std():.1f}pp")


In [ ]:

# ── 6.4  Missing row gap attribution — §4.4 ──────────────────────────────────
NOMINAL_PER_RUN = 510 * 41  # 20,910

missing_summary_rows = []
for label in BENCHMARK_MODELS:
    ext = RAW[label]['ext']
    for run in sorted(ext['run_id'].unique()):
        run_df = ext[ext['run_id'] == run]
        contract_sizes = run_df.groupby('title').size()
        incomplete = int((contract_sizes < 41).sum())
        missing_summary_rows.append({
            'Model': label,
            'Run': int(run),
            'Actual rows': len(run_df),
            'Missing rows': NOMINAL_PER_RUN - len(run_df),
            'Incomplete contracts': incomplete,
        })

missing_df = pd.DataFrame(missing_summary_rows)
display_title('Missing row analysis — §4.4 (is the gap random, run-correlated, or contract-correlated?)')
display(missing_df.style.hide(axis='index'))

# Classify gap pattern per model
print("\nGap attribution per model:")
for label in BENCHMARK_MODELS:
    sub = missing_df[missing_df['Model'] == label]
    total_missing = int(sub['Missing rows'].sum())
    mean_miss = sub['Missing rows'].mean()
    run_cv = sub['Missing rows'].std() / (mean_miss + 1e-9)
    pattern = 'run-correlated (variable across runs)' if run_cv > 0.3 else 'approximately uniform (random / contract-level)'
    print(f"  {label}: total_missing={total_missing:,}, run_cv={run_cv:.2f} → {pattern}")

# Overlap analysis for qwen3-32b (largest gap — 1,194 rows)
print("\n── qwen3-32b overlap analysis ──")
qwen_ext = RAW['qwen3-32b']['ext']
qwen_incomplete: dict[int, set] = {}
for run in sorted(qwen_ext['run_id'].unique()):
    run_df = qwen_ext[qwen_ext['run_id'] == run]
    sizes = run_df.groupby('title').size()
    qwen_incomplete[int(run)] = set(sizes[sizes < 41].index)

runs = list(qwen_incomplete.keys())
overlap_all = qwen_incomplete[runs[0]].intersection(*[qwen_incomplete[r] for r in runs[1:]])
union_all   = qwen_incomplete[runs[0]].union(*[qwen_incomplete[r] for r in runs[1:]])
frac = len(overlap_all) / max(len(union_all), 1)
print(f"Incomplete in ALL 3 runs: {len(overlap_all)} contracts")
print(f"Incomplete in ANY run:    {len(union_all)} contracts")
print(f"Persistent fraction (contract-correlated): {frac:.1%}")
print("=> >50% → structural (contract-level bias); <20% → random API noise")


In [ ]:

# ── 6.5  Composite score rank sensitivity to weighting scheme ─────────────────
WEIGHT_SCHEMES = {
    'Equal (1:1:1:1:1)':           dict(w_far=1, w_frr=1, w_acc=1, w_hal=1, w_jeq=1),
    'FP-heavy (2:1:1:2:1)':        dict(w_far=2, w_frr=1, w_acc=1, w_hal=2, w_jeq=1),
    'Recall-heavy (1:2:1:1:2)':    dict(w_far=1, w_frr=2, w_acc=1, w_hal=1, w_jeq=2),
    'Halluc-only (0:0:0:1:0)':     dict(w_far=0, w_frr=0, w_acc=0, w_hal=1, w_jeq=0),
    'Detection-only (1:1:1:0:0)':  dict(w_far=1, w_frr=1, w_acc=1, w_hal=0, w_jeq=0),
}


def rank_with_weights(df, halluc_col, w_far, w_frr, w_acc, w_hal, w_jeq):
    work = df.copy()
    total_w = (w_far + w_frr + w_acc + w_hal + w_jeq) or 1
    work['_score'] = (
        work['far'].rank(ascending=True) * w_far
        + work['frr'].rank(ascending=True) * w_frr
        + work['det_acc'].rank(ascending=False) * w_acc
        + work[halluc_col].rank(ascending=True) * w_hal
        + work['judge_eq'].rank(ascending=False) * w_jeq
    ) / total_w
    work['_rank'] = work['_score'].rank(method='min').astype(int)
    return work.set_index('label')['_rank'].to_dict()


sensitivity_rows = []
for scheme, weights in WEIGHT_SCHEMES.items():
    ranks = rank_with_weights(subset_overall, 'halluc_generated', **weights)
    sensitivity_rows.append({'Scheme': scheme, **ranks})

sensitivity_df = pd.DataFrame(sensitivity_rows).set_index('Scheme')[SUBSET_MODELS]

display_title('Table S4 — Composite leaderboard rank sensitivity to weighting (Exp. 2 subset)')
display(
    sensitivity_df.style
    .background_gradient(cmap='RdYlGn_r', vmin=1, vmax=len(SUBSET_MODELS), axis=None)
    .format('{:d}')
)

is_robust = all(sensitivity_df['gemma-4-26b (debate)'].values == 1)
print(f"\ngemma-4-26b (debate) rank-1 in ALL schemes: {is_robust}")
if not is_robust:
    print("Schemes where it is NOT rank-1:")
    print(sensitivity_df[sensitivity_df['gemma-4-26b (debate)'] != 1]['gemma-4-26b (debate)'].to_string())


In [ ]:

# ── 6.6  Debate pipeline overhead proxy ──────────────────────────────────────
debate_meta = RAW['gemma-4-26b (debate)']['ext']

rounds_by_type = (
    debate_meta.groupby('clause_type')['debate_rounds']
    .agg(mean_rounds='mean', std_rounds='std', n='count')
    .round(3)
    .rename(columns={'mean_rounds': 'Mean rounds', 'std_rounds': 'SD rounds', 'n': 'N'})
)
display_title('Debate rounds by claim type (pipeline overhead proxy)')
display(rounds_by_type)

print(f"\nOverall debate_rounds: mean={debate_meta['debate_rounds'].mean():.2f}, "
      f"median={debate_meta['debate_rounds'].median():.0f}, "
      f"max={debate_meta['debate_rounds'].max():.0f}")
print("\nValue counts:")
print(debate_meta['debate_rounds'].value_counts().sort_index().to_string())

if 'debate_changed' in debate_meta.columns:
    changed_overall = float(debate_meta['debate_changed'].mean() * 100)
    changed_by_type = debate_meta.groupby('clause_type')['debate_changed'].mean().mul(100).round(1)
    print(f"\nDetection flipped by debate (overall): {changed_overall:.1f}%")
    print("Flip rate by claim type:")
    print(changed_by_type.to_string())

if 'debate_consensus' in debate_meta.columns:
    print("\nDebate consensus distribution:")
    print(debate_meta['debate_consensus'].value_counts(normalize=True).mul(100).round(1).to_string())

if 'debate_stability' in debate_meta.columns:
    print("\nDebate stability distribution:")
    print(debate_meta['debate_stability'].value_counts(normalize=True).mul(100).round(1).to_string())

debate_overhead_df = (
    debate_meta.groupby('clause_type')['debate_rounds']
    .agg(mean_rounds='mean', std_rounds='std', n='count')
    .round(3)
    .reset_index()
    .rename(columns={'clause_type': 'Claim type', 'mean_rounds': 'Mean rounds', 'std_rounds': 'SD rounds', 'n': 'N'})
)


## 6. Figures

The figures below are exported to `Notebooks/batch_results/comparisons/figures/` with stable paper-oriented filenames.

In [ ]:
# Figure 1 — benchmark typed hallucination profiles
bench_plot = benchmark_typed.pivot(index='label', columns='clause_type', values='halluc_tp').loc[BENCHMARK_MODELS, CLAIM_ORDER_FOR_PAPER]
fig, ax = plt.subplots(figsize=(14, 6))
bar_width = 0.18
x = np.arange(len(BENCHMARK_MODELS))
claim_palette = {
    'numeric': '#d63031',
    'obligation_entitlement': '#e17055',
    'factual': '#74b9ff',
    'temporal': '#00b894',
}
for idx, claim_type in enumerate(CLAIM_ORDER_FOR_PAPER):
    vals = bench_plot[claim_type].values
    bars = ax.bar(x + (idx - 1.5) * bar_width, vals, bar_width, label=CLAIM_LABELS[claim_type], color=claim_palette[claim_type], edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4, f'{val:.1f}%', ha='center', va='bottom', fontsize=8, rotation=90)
ax.set_xticks(x)
ax.set_xticklabels(BENCHMARK_MODELS, rotation=15, ha='right')
ax.set_ylabel('HallucTP %')
ax.set_title('Figure 1 — Experiment 1 typed hallucination benchmark')
ax.set_ylim(0, 85)
ax.legend(ncol=2, frameon=True)
plt.tight_layout()
fig1_path = FIG_DIR / 'fig01_experiment1_typed_hallucination.png'
plt.savefig(fig1_path, dpi=180, bbox_inches='tight')
plt.show()

# Figure 2 — benchmark direction stacked bars with RDI annotation
fig, ax = plt.subplots(figsize=(12, 6))
plot_df = benchmark_direction_overall.set_index('label').loc[BENCHMARK_MODELS]
bottom = np.zeros(len(plot_df))
stack_parts = [
    ('scope_pct', 'Scope', '#95a5a6'),
    ('missing_pct', 'Missing condition', '#f1c40f'),
    ('extra_pct', 'Extra condition', '#e84393'),
]
for col, label, color in stack_parts:
    ax.bar(plot_df.index, plot_df[col], bottom=bottom, label=label, color=color, edgecolor='white')
    bottom += plot_df[col].values
for idx, (_, row) in enumerate(plot_df.iterrows()):
    ax.text(idx, 103, f"RDI {row['rdi']:+.3f}", ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('% of contradicted TP findings')
ax.set_ylim(0, 110)
ax.set_title('Figure 2 — Experiment 1 error direction and RDI')
ax.legend(frameon=True, ncol=3, loc='upper center')
plt.xticks(rotation=10)
plt.tight_layout()
fig2_path = FIG_DIR / 'fig02_experiment1_direction_rdi.png'
plt.savefig(fig2_path, dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3 — mitigation leaderboard on matched subset
fig, ax = plt.subplots(figsize=(11, 6))
leaderboard_plot = subset_leaderboard.copy()
y = np.arange(len(leaderboard_plot))
colors = [MODEL_REGISTRY[label]['color'] for label in leaderboard_plot['label']]
ax.barh(y, leaderboard_plot['Score'], color=colors, edgecolor='white')
for yi, (_, row) in zip(y, leaderboard_plot.iterrows()):
    ax.text(row['Score'] + 0.03, yi, f"Score {row['Score']:.2f} | HallucGen {row['halluc_generated']:.1f}% | JEq {row['judge_eq']:.1f}%", va='center', fontsize=9)
ax.set_yticks(y)
ax.set_yticklabels(leaderboard_plot['label'])
ax.invert_yaxis()
ax.set_xlabel('Composite score (lower is better)')
ax.set_title('Figure 3 — Experiment 2 mitigation leaderboard on the matched subset')
plt.tight_layout()
fig3_path = FIG_DIR / 'fig03_experiment2_matched_leaderboard.png'
plt.savefig(fig3_path, dpi=180, bbox_inches='tight')
plt.show()

# Figure 4 — gemma mitigation deltas by claim type
plot_delta = debate_delta[debate_delta['Claim type'] != 'ALL'].copy()
x = np.arange(len(plot_delta))
bar_width = 0.2
fig, ax = plt.subplots(figsize=(12, 6))
series = [
    ('ΔFAR', '#d63031'),
    ('ΔAcc', '#00b894'),
    ('ΔHallucGen', '#6c5ce7'),
    ('ΔJudgeEq', '#0984e3'),
]
for idx, (col, color) in enumerate(series):
    vals = plot_delta[col].values
    ax.bar(x + (idx - 1.5) * bar_width, vals, bar_width, label=col, color=color, edgecolor='white')
ax.axhline(0, color='black', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(plot_delta['Claim type'])
ax.set_ylabel('Delta (percentage points)')
ax.set_title('Figure 4 — Gemma mitigation deltas by claim type')
ax.legend(frameon=True, ncol=4)
plt.tight_layout()
fig4_path = FIG_DIR / 'fig04_experiment2_debate_deltas.png'
plt.savefig(fig4_path, dpi=180, bbox_inches='tight')
plt.show()

# Figure 5 — RDI shift under mitigation
plot_rdi = rdi_shift.copy()
x = np.arange(len(plot_rdi))
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(x, plot_rdi['Baseline RDI'], marker='o', linewidth=2, color='#e17055', label='Gemma baseline')
ax.plot(x, plot_rdi['Debate RDI'], marker='o', linewidth=2, color='#00b894', label='Gemma debate')
for xi, (_, row) in zip(x, plot_rdi.iterrows()):
    ax.vlines(xi, row['Baseline RDI'], row['Debate RDI'], color='#b2bec3', linewidth=2, alpha=0.8)
ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(plot_rdi['Claim type'])
ax.set_ylabel('RDI')
ax.set_title('Figure 5 — Gemma RDI shift under mitigation')
ax.legend(frameon=True)
plt.tight_layout()
fig5_path = FIG_DIR / 'fig05_experiment2_rdi_shift.png'
plt.savefig(fig5_path, dpi=180, bbox_inches='tight')
plt.show()

## 7. Export Bundle

This final cell writes the paper-ready CSV exports and a manifest describing the generated artifacts.

In [ ]:

import datetime

def _df_to_records(df: pd.DataFrame) -> list:
    """Serialise a DataFrame to a JSON-safe list of dicts."""
    return json.loads(df.to_json(orient='records', double_precision=4, date_format='iso'))


# ── Build the single consolidated results JSON ────────────────────────────────
results = {
    'meta': {
        'generated_at': datetime.datetime.now().isoformat(timespec='seconds'),
        'benchmark_nominal_per_model': 510 * 41 * 3,
        'benchmark_observed_total': int(benchmark_overall['n'].sum()),
        'subset_contracts': 120,
        'subset_clause_types': 41,
        'n_benchmark_models': len(BENCHMARK_MODELS),
        'n_subset_models': len(SUBSET_MODELS),
        'bootstrap_resamples': N_BOOT,
        'figures': [fig1_path.name, fig2_path.name, fig3_path.name,
                    fig4_path.name, fig5_path.name],
    },

    # ── Experiment 1 ─────────────────────────────────────────────────────────
    'experiment1': {
        'description': 'Full benchmark — 510 contracts × 41 clauses × 3 runs, 4 models',
        'table1_overall': _df_to_records(benchmark_table1),
        'table2_typed_hallucTP': _df_to_records(benchmark_table2),
        'table3_direction_rdi': _df_to_records(benchmark_table3),
        'typed_rdi_by_model': _df_to_records(benchmark_typed_rdi),
        'full_metrics': _df_to_records(benchmark_metrics),
        'integrity_checks': _df_to_records(benchmark_checks),
        'support_calculations': {
            'qwen3_32b': {
                'missing_pct': round(float(qwen_overall_dir['missing_pct']), 2),
                'extra_pct':   round(float(qwen_overall_dir['extra_pct']),   2),
                'rdi':         round(float(qwen_overall_dir['rdi']),          3),
            },
            'gpt_5_2': {
                'missing_pct': round(float(gpt_overall_dir['missing_pct']), 2),
                'extra_pct':   round(float(gpt_overall_dir['extra_pct']),   2),
                'rdi':         round(float(gpt_overall_dir['rdi']),          3),
            },
            'llama_3_3_70b_numeric_abstention': {
                'frr':       round(float(llama_numeric['frr']),       2),
                'judge_eq':  round(float(llama_numeric['judge_eq']),  2),
                'halluc_tp': round(float(llama_numeric['halluc_tp']), 2),
            },
        },
    },

    # ── Experiment 2 ─────────────────────────────────────────────────────────
    'experiment2': {
        'description': 'Matched 120-contract subset — gemma baseline vs debate, single run',
        'table4_leaderboard': _df_to_records(
            subset_table4.drop(columns=['#'], errors='ignore')
        ),
        'table5_debate_deltas': _df_to_records(debate_delta),
        'table6_rdi_shift': _df_to_records(rdi_shift),
        'full_metrics': _df_to_records(subset_metrics),
        'direction_metrics': _df_to_records(subset_direction),
    },

    # ── Section 6 reviewer additions ─────────────────────────────────────────
    'reviewer_additions': {

        's1_per_run_variance': {
            'description': 'Mean ± SD across 3 runs for all metrics (MW #2)',
            'records': _df_to_records(variance_df),
            'overall_summary': _df_to_records(paper_var),
            'typed_hallucTP_sd': _df_to_records(halluc_sd_pivot),
        },

        's2_rdi_bootstrap_ci': {
            'description': '95% bootstrap CIs on RDI, 2000 resamples, all runs pooled (MW #2)',
            'n_bootstrap': N_BOOT,
            'records': _df_to_records(rdi_ci_df),
            'overall_table': _df_to_records(
                rdi_ci_df[rdi_ci_df['clause_type'] == 'ALL']
                [['label', 'rdi', 'ci_lo', 'ci_hi', 'rdi_ci_str']]
                .rename(columns={'label': 'Model'})
            ),
        },

        's3_obligation_subtype_profiles': {
            'description': 'HallucTP & FAR per obligation/entitlement subtype (MW #6)',
            'n_subtypes': len(clause_rank_order),
            'subtype_order_by_mean_hallucTP': clause_rank_order,
            'hallucTP_range_pp': {
                'min': round(float(halluc_tp_pivot['Mean'].min()), 1),
                'max': round(float(halluc_tp_pivot['Mean'].max()), 1),
            },
            'within_bucket_sd_pp': round(
                float(oblig_subtype_df.groupby('clause_name')['halluc_tp'].mean().std()), 1
            ),
            'records': _df_to_records(oblig_subtype_df),
        },

        's4_missing_row_attribution': {
            'description': 'Per-run missing rows — random vs run vs contract correlated (§4.4)',
            'nominal_per_run': NOMINAL_PER_RUN,
            'records': _df_to_records(missing_df),
            'qwen_overlap': {
                'incomplete_in_all_3_runs': len(overlap_all),
                'incomplete_in_any_run':    len(union_all),
                'persistent_fraction':      round(frac, 3),
            },
        },

        's5_composite_rank_sensitivity': {
            'description': 'Leaderboard rank under 5 weighting schemes (Minor §6.2)',
            'schemes': list(WEIGHT_SCHEMES.keys()),
            'gemma_debate_rank1_in_all_schemes': bool(is_robust),
            'records': json.loads(
                sensitivity_df.reset_index().to_json(orient='records')
            ),
        },

        's6_debate_overhead': {
            'description': 'Debate rounds distribution as cost proxy (Minor §3.3)',
            'mean_rounds_overall': round(float(debate_meta['debate_rounds'].mean()), 3),
            'median_rounds_overall': int(debate_meta['debate_rounds'].median()),
            'max_rounds': int(debate_meta['debate_rounds'].max()),
            'pct_changed_detection': round(changed_overall, 2),
            'rounds_by_type': _df_to_records(
                debate_meta.groupby('clause_type')['debate_rounds']
                .agg(mean_rounds='mean', std_rounds='std', n='count')
                .round(3).reset_index()
            ),
            'flip_rate_by_type': json.loads(
                debate_meta.groupby('clause_type')['debate_changed']
                .mean().mul(100).round(2).reset_index()
                .rename(columns={'debate_changed': 'flip_pct'})
                .to_json(orient='records')
            ) if 'debate_changed' in debate_meta.columns else None,
            'consensus_distribution': (
                debate_meta['debate_consensus']
                .value_counts(normalize=True).mul(100).round(2).to_dict()
                if 'debate_consensus' in debate_meta.columns else None
            ),
            'stability_distribution': (
                debate_meta['debate_stability']
                .value_counts(normalize=True).mul(100).round(2).to_dict()
                if 'debate_stability' in debate_meta.columns else None
            ),
        },
    },
}

RESULTS_JSON = OUT_DIR / 'paper_all_results.json'
with open(RESULTS_JSON, 'w') as f:
    json.dump(results, f, indent=2, allow_nan=False,
              default=lambda o: None if (isinstance(o, float) and (o != o)) else str(o))

print(f'Written: {RESULTS_JSON}')
print(f'Top-level keys: {list(results.keys())}')
print(f'reviewer_additions keys: {list(results["reviewer_additions"].keys())}')
print(f'\nFile size: {RESULTS_JSON.stat().st_size / 1024:.1f} KB')
